# Polar-only chip creation

This is the polar-only counterpart to [chip_example.ipynb](chip_example.ipynb), using the same chip backend. Start with one small north- or south-polar AOI, away from ±82° and the pole itself. Polar antimeridian AOIs are supported: west=179.9, east=-179.9 describes a small wrapped region, not a near-global chip. Cross-82° and pole-containing chips remain unsupported.

The HPC regression suites passed (269 chip tests and 29 targeted tiling tests). A real-data Run All and overlay inspection of this notebook remain pending. Edit the source/label paths and AOI together; the example AOI does not guarantee that the latest label export covers it.

Dynamic output retains the original raster CRS and pixel lattice. Static-only output uses a 100 m, zero-anchored lattice in the hemisphere's repository LPS CRS. Acquisition tiles are intermediate, not the output grid. Select the project's GDAL-enabled kernel.


In [ ]:
from pathlib import Path
from datetime import datetime
import sys

repo_root = Path(str(Path.cwd().parent).replace(
    "/panfs/ccds02/nobackup", "/explore/nobackup"))
if not (repo_root / "lfm/data_processing").is_dir():
    raise FileNotFoundError("Run from the repository's top-level notebooks/ directory.")
sys.path.insert(0, str(repo_root))

from lfm.data_processing.chip import (
    AcquisitionGroupConfig, ChipConfig, GeographicAOI, LabelInput,
    NoSplitConfig, OutputModalityConfig, SourceSelector,
    chip_grid_family, default_chip_zoom, chip_request_from_aoi, create_chips,
)
from lfm.data_processing.tiling import (
    TileConfig, TileSourceConfig, VectorIndexBuildConfig,
    ensure_vector_index, resolve_notebook_source_index, discover_raster_paths,
)
from lfm.data_processing.tiling.product_ids import lunar_product_id_from_raster_path
from lfm.all_models.all_tasks.tiling_utils import make_static_source
from lfm.data_processing.chip.chip_notebook_utils import (
    latest_crater_label_path, read_source_grid, plot_chip_result,
)


## 1. Configure one study area

- `HEMISPHERE`: north/south example; `AOI_NWSE` uses IAU:30100 north, west, south, east. The north default encloses craters 13–20 in the supplied `(2).gpkg`; the south example requires its own labels.
- `MODALITY`: `wac`, `nac`, or `static`. Static-only ignores the dynamic raster; otherwise `INCLUDE_STATIC` appends the canonical 63 bands.
- `SOURCE_RASTER`: original pre-tiling raster (TIFF or VRT). For the default WAC example, `None` discovers exactly one hemisphere-specific VRT in the tiling example's collection. NAC requires an explicit path.
- `WAC_LAYOUT`: `single` for the polar monochrome mosaic, or `vis_uv` for the usual seven-channel VIS/UV product. The selected VIS reference has five bands for `vis_uv`; its UV companion must be available.
- `DYNAMIC_GLOB`: optional companion-file pattern; by default index only the selected raster. For `vis_uv`, derive the product's VIS/UV TIFF pattern. Never index both a VRT and its backing TIFF as duplicate imagery.
- `LABEL_PATH`: explicit GPKG/TIFF, or `None` for the latest crater-notebook GPKG. Confirm the printed path; discovery does not match hemisphere/product. `LABEL_KIND`: `auto` (TIFF semantic), `semantic`, or `raster_instance`. `LABEL_LAYER` applies only to GPKG.
- `ZOOM_OVERRIDE`: acquisition override only; defaults are WAC/static 4, NAC 10. Final output resolution still follows the native source or static-only 100 m grid.
- `MAX_OUTPUT_PIXELS`: safety limit before indexing/acquisition; reduce the AOI for high-resolution NAC. `MAX_WORKERS`: chip workers, not index workers.


In [ ]:
PROJECT_DATA_DIR = Path("/explore/nobackup/projects/lfm")
HEMISPHERE = "north"  # "north" or "south".
MODALITY = "wac"  # "wac", "nac", or "static".
INCLUDE_STATIC = False  # Start with imagery only; verify static coverage separately.
SOURCE_RASTER = None  # Set an explicit Path for your source, especially NAC.
WAC_LAYOUT = "single"  # Polar mosaic is not the seven-band VIS/UV product.
DYNAMIC_GLOB = None  # Optional glob selecting this product's companion files.
POLAR_WAC_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/Static_final/LROC/WAC/wac_glob_morf_mos"
STATIC_DATA_DIR = PROJECT_DATA_DIR / "staticLinks"
AOI_NWSE = (86.1, -48.7, 83.95, -19.5) if HEMISPHERE == "north" else (-85.9, -0.1, -86.1, 0.1)
# Antimeridian alternative; confirm your imagery and labels cover this area:
# AOI_NWSE = (86.01, 179.98, 86.0, -179.98) if HEMISPHERE == 'north' else (-86.0, 179.98, -86.01, -179.98)
LABEL_PATH = repo_root / "test_outputs" / "WAC_GLOBAL_P900N0000_100M.eqc.iau2.LPS_N_label_craters (2).gpkg"  # Set your HPC copy's path if different.
LABEL_KIND = "auto"  # For instance TIFFs use "raster_instance".
LABEL_LAYER = "craters"
SAMPLE_ID = f"{MODALITY}_{HEMISPHERE}_aoi_1"  # Unique per study area.
ZOOM_OVERRIDE = None
MAX_OUTPUT_PIXELS = 4_000_000  # Bounds per-band memory; static adds 63 bands.
MAX_WORKERS = 1
INDEX_WORKER_COUNT = 1
SPLIT_CONFIG = NoSplitConfig(seed=42)
OUTPUT_BASE_DIR = repo_root / "notebooks/outputs/chip_creation_polar"

### Optional full workflow

After validating the single example, replace the request list with multiple AOIs on the same source lattice and in the same hemisphere. Give each AOI a unique ID. Keep a shared product split-group key to prevent train/test leakage. Separate runs are needed for different source grids or acquisition zooms.


In [ ]:
# from lfm.data_processing.chip import MixedPercentageNumberSplitConfig, SplitCounts, SplitPercentages
# SPLIT_CONFIG = MixedPercentageNumberSplitConfig(
#     fixed_counts=SplitCounts(test=100), remaining_percentages=SplitPercentages(train=0.9, val=0.1),
#     seed=42,
# )
# MAX_WORKERS = 16
# Below, replace requests = [request] with requests built for your AOIs.


## 2. Resolve labels and inspect the output grid

No imagery acquisition occurs here. The realized rectangular raster may extend slightly beyond the input AOI due to reprojection and outward pixel rounding; labels must cover the entire realized grid. Review dimensions and label association before continuing. Canonical static coverage can be partial or absent; verified empty channels retain -32768 with warnings, while corrupt/missing inputs still fail.


In [ ]:
if HEMISPHERE not in ("north", "south") or MODALITY not in ("wac", "nac", "static"):
    raise ValueError("Choose a supported hemisphere and modality.")
if WAC_LAYOUT not in ("single", "vis_uv"):
    raise ValueError("WAC_LAYOUT must be 'single' or 'vis_uv'.")
requested_aoi = GeographicAOI(*AOI_NWSE)
family = chip_grid_family(requested_aoi)
expected_family = "lps_n" if HEMISPHERE == "north" else "lps_s"
if family.value != expected_family:
    raise ValueError("AOI must lie wholly in the selected polar hemisphere.")
static_only = MODALITY == "static"
source_grid = None
selectors = ()
GROUP_NAME = "polar"
if not static_only:
    if SOURCE_RASTER is None:
        if MODALITY != "wac" or WAC_LAYOUT != "single":
            raise ValueError("Set SOURCE_RASTER to your original imagery raster.")
        matches = sorted(POLAR_WAC_DIR.glob(f"*{family.value.upper()}.vrt"))
        if len(matches) != 1:
            raise ValueError(f"Expected one polar VRT; found {len(matches)}. Set SOURCE_RASTER explicitly.")
        SOURCE_RASTER = matches[0]
    SOURCE_RASTER = Path(SOURCE_RASTER)
    source_grid = read_source_grid(
        SOURCE_RASTER, expected_band_count=5 if MODALITY == "wac" and WAC_LAYOUT == "vis_uv" else 1)
    PRODUCT_ID = lunar_product_id_from_raster_path(SOURCE_RASTER)
    selectors = (SourceSelector(GROUP_NAME, MODALITY, PRODUCT_ID),)
LABEL_PATH = latest_crater_label_path() if LABEL_PATH is None else Path(LABEL_PATH)
if not LABEL_PATH.is_file():
    raise FileNotFoundError(LABEL_PATH)
label_input = LabelInput(LABEL_PATH, kind=LABEL_KIND, relation="clip_to_target",
                         layer=LABEL_LAYER if LABEL_PATH.suffix.lower() == ".gpkg" else None)
request = chip_request_from_aoi(
    sample_id=SAMPLE_ID, geographic_aoi=requested_aoi,
    source_grid=source_grid, static_only=static_only,
    split_group_key=SAMPLE_ID if static_only else PRODUCT_ID,
    source_selectors=selectors, label_input=label_input,
)
if request.target_grid.width * request.target_grid.height > MAX_OUTPUT_PIXELS:
    raise ValueError("Output exceeds MAX_OUTPUT_PIXELS; reduce AOI or explicitly raise the limit.")
ZOOM_LEVEL = default_chip_zoom(request.geographic_aoi, MODALITY) if ZOOM_OVERRIDE is None else ZOOM_OVERRIDE
OUTPUT_ROOT = OUTPUT_BASE_DIR / datetime.now().strftime("%Y%m%dT%H%M%S_%f")
print(f"Labels: {LABEL_PATH}")
print(f"Grid family: {family.value} | acquisition zoom: {ZOOM_LEVEL}")
print(f"Output: {request.target_grid.width} x {request.target_grid.height} | affine: {request.target_grid.transform}")
print(f"Destination: {OUTPUT_ROOT}")


## 3. Prepare enabled indexes and configure chips

Dynamic indexing is restricted to the chosen file/product and stored in this run's directory. Shared static indexes remain read-only; custom static collections use a per-clone cache. Disabled collections are not inspected. Single-band polar WAC is explicitly selected as one band, never forced into the seven-band contract.


In [ ]:
sources, modalities = [], []
if not static_only:
    pattern = DYNAMIC_GLOB or (
        f"{PRODUCT_ID}.prj.*.mos.tif" if MODALITY == "wac" and WAC_LAYOUT == "vis_uv"
        else SOURCE_RASTER.name)
    index_config = VectorIndexBuildConfig(
        data_dir=SOURCE_RASTER.parent, index_path=OUTPUT_ROOT / "indexes/dynamic.gpkg",
        image_glob=pattern, worker_count=INDEX_WORKER_COUNT)
    if SOURCE_RASTER.resolve() not in {p.resolve() for p in discover_raster_paths(index_config)}:
        raise ValueError('DYNAMIC_GLOB must include SOURCE_RASTER.')
    index = ensure_vector_index(index_config, stdout=sys.stdout).index_path
    sources.append(TileSourceConfig(
        name=MODALITY, data_dir=SOURCE_RASTER.parent, index_path=index,
        selection_mode="product_id", preserve_source_nodata=True, required=True))
    single = MODALITY == "nac" or WAC_LAYOUT == "single"
    modalities.append(OutputModalityConfig(
        GROUP_NAME, MODALITY, MODALITY,
        band_indices=(1,) if single else None,
        output_band_names=(MODALITY,) if single else None))
if static_only or INCLUDE_STATIC:
    resolution = resolve_notebook_source_index(
        source_name="static", data_dir=STATIC_DATA_DIR, cache_dir=OUTPUT_BASE_DIR / "indexes")
    index = ensure_vector_index(VectorIndexBuildConfig(
        data_dir=resolution.data_dir, index_path=resolution.index_path,
        rebuild_invalid_index=resolution.rebuild_invalid_index,
        worker_count=INDEX_WORKER_COUNT), stdout=sys.stdout).index_path
    sources.append(make_static_source(data_dir=STATIC_DATA_DIR, index_path=index))
    modalities.append(OutputModalityConfig(GROUP_NAME, "static", "static"))
chip_config = ChipConfig(
    output_root=OUTPUT_ROOT, label_source=LABEL_PATH,
    acquisition_groups=(AcquisitionGroupConfig(
        GROUP_NAME, TileConfig(OUTPUT_ROOT / ".tiling-template", ZOOM_LEVEL, tuple(sources))),),
    output_modalities=tuple(modalities), split_config=SPLIT_CONFIG,
)
requests = [request]


## 4. Create and inspect

Per-sample errors are reported without silently treating failures as successful chips. Review the manifest, warnings and overlay. Run north/south, dynamic-only/mixed/static-only cases separately before relying on a new collection.


In [ ]:
batch = create_chips(requests, chip_config, max_workers=MAX_WORKERS)
print(f"Manifest: {batch.manifest_path}")
for result in batch.results:
    print(result.request.sample_id, result.status, result.message)
    for diagnostic in result.diagnostics:
        print(f"{diagnostic.stage}/{diagnostic.code}: {diagnostic.message}")
    if result.status == "success":
        plot_chip_result(result, display_band_keyword="elv" if static_only else MODALITY)
